In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import numpy as np

AWKERNEL_DIR = "awkernel_results"
BENCHMARK_DIR = "benchmark_results/20260227_130120"

# TU mapping: awkernel TU10 -> 1.0, TU15 -> 1.5, etc.
TU_VALUES = [1.0, 1.5, 2.0, 2.5, 3.0]
# TU values to use in box plot (exclude 1.5 and 2.5)
TU_BOX = [tu for tu in TU_VALUES if tu not in (1.5, 2.5)]

# Collect awkernel latency data (ns -> us) and deadline miss info
awkernel_data = {}
awkernel_deadline_miss = {}
for tu in TU_VALUES:
    tu_int = int(tu * 10)
    path = os.path.join(AWKERNEL_DIR, f"RD-Gen(GEDF3)_TU{tu_int}.csv")
    if os.path.exists(path):
        df = pd.read_csv(path)
        latency = pd.to_numeric(df["Latency"], errors="coerce")
        deadline = pd.to_numeric(df["Relative Deadline"], errors="coerce")
        valid = latency.notna() & deadline.notna()
        awkernel_data[tu] = latency[valid].values / 1000  # ns -> us
        awkernel_deadline_miss[tu] = (latency[valid] > deadline[valid]).mean()

# Collect benchmark response_time_us data and deadline miss info
benchmark_data = {}
benchmark_deadline_miss = {}
for tu in TU_VALUES:
    tu_dir = os.path.join(BENCHMARK_DIR, f"TU_{tu}")
    if not os.path.isdir(tu_dir):
        continue
    all_rt = []
    all_dl = []
    for case_dir in sorted(glob.glob(os.path.join(tu_dir, "case_*"))):
        for csv_file in glob.glob(os.path.join(case_dir, "response_time_*.csv")):
            df = pd.read_csv(csv_file)
            all_rt.extend(df["response_time_us"].values)
            all_dl.extend(df["deadline_us"].values)
    if all_rt:
        all_rt = np.array(all_rt)
        all_dl = np.array(all_dl)
        benchmark_data[tu] = all_rt
        benchmark_deadline_miss[tu] = (all_rt > all_dl).mean()

In [ ]:
plt.rcParams.update({"font.size": 16})

fig, ax = plt.subplots(figsize=(6, 6))

positions = np.arange(len(TU_BOX))
width = 0.35

# Prepare box plot data (only TU_BOX values)
awkernel_box_data = [awkernel_data.get(tu, []) for tu in TU_BOX]
benchmark_box_data = [benchmark_data.get(tu, []) for tu in TU_BOX]

flier_common = dict(marker=".", markersize=4, markeredgecolor="black", markerfacecolor="black")

bp1 = ax.boxplot(
    awkernel_box_data,
    positions=positions - width / 2,
    widths=width * 0.8,
    patch_artist=True,
    boxprops=dict(facecolor="white", edgecolor="black", linewidth=1),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1),
    capprops=dict(color="black", linewidth=1),
    flierprops=dict(**flier_common),
)

bp2 = ax.boxplot(
    benchmark_box_data,
    positions=positions + width / 2,
    widths=width * 0.8,
    patch_artist=True,
    boxprops=dict(facecolor="lightgray", edgecolor="black", linewidth=1, hatch="//"),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1),
    capprops=dict(color="black", linewidth=1),
    flierprops=dict(**flier_common),
)

ax.set_xticks(positions)
ax.set_xticklabels([f"{tu}" for tu in TU_BOX])
ax.set_xlabel("Total Utilization")
ax.set_ylabel("Response Time [$\\mu$s]")
ax.set_title("Response Time Comparison")
ax.legend(
    [bp1["boxes"][0], bp2["boxes"][0]],
    ["awkernel", "CIE + SCHED_DEADLINE"],
    fontsize=13,
)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("response_time_comparison.pdf", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

aw_miss = [awkernel_deadline_miss.get(tu, np.nan) * 100 for tu in TU_VALUES]
bm_miss = [benchmark_deadline_miss.get(tu, np.nan) * 100 for tu in TU_VALUES]

ax.plot(TU_VALUES, aw_miss, marker="o", label="awkernel", color="#4C72B0")
ax.plot(TU_VALUES, bm_miss, marker="s", label="benchmark", color="#DD8452")

ax.set_xlabel("Total Utilization")
ax.set_ylabel("Deadline Miss Ratio [%]")
ax.set_title("Deadline Miss Ratio Comparison: awkernel vs benchmark")
ax.set_xticks(TU_VALUES)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("deadline_miss_ratio_comparison.png", dpi=150)
plt.show()